In [1]:
import numpy as np
import os
import cv2

class Layer_Dense:
  def __init__(self, n_inputs, n_neurons,
  weight_regularizer_l1=0, weight_regularizer_l2=0,
  bias_regularizer_l1=0, bias_regularizer_l2=0):
    self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
    self.biases = np.zeros((1, n_neurons))
    self.weight_regularizer_l1 = weight_regularizer_l1
    self.weight_regularizer_l2 = weight_regularizer_l2
    self.bias_regularizer_l1 = bias_regularizer_l1
    self.bias_regularizer_l2 = bias_regularizer_l2
  def forward(self, inputs, training):
    self.inputs = inputs
    self.output = np.dot(inputs, self.weights) + self.biases
  def backward(self, dvalues):
    self.dweights = np.dot(self.inputs.T, dvalues)
    self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
    if self.weight_regularizer_l1 > 0:
      dL1 = np.ones_like(self.weights)
      dL1[self.weights < 0] = -1
      self.dweights += self.weight_regularizer_l1 * dL1
    if self.weight_regularizer_l2 > 0:
      self.dweights += 2 * self.weight_regularizer_l2 * \
      self.weights
    if self.bias_regularizer_l1 > 0:
      dL1 = np.ones_like(self.biases)
      dL1[self.biases < 0] = -1
      self.dbiases += self.bias_regularizer_l1 * dL1
    if self.bias_regularizer_l2 > 0:
      self.dbiases += 2 * self.bias_regularizer_l2 * \
      self.biases
    self.dinputs = np.dot(dvalues, self.weights.T)

class Layer_Input:
  def forward(self, inputs, training):
    self.output = inputs

In [2]:
class Loss:
  def regularization_loss(self):
    regularization_loss = 0
    for layer in self.trainable_layers:
      if layer.weight_regularizer_l1 > 0:
        regularization_loss += layer.weight_regularizer_l1 * \
        np.sum(np.abs(layer.weights))
      if layer.weight_regularizer_l2 > 0:
        regularization_loss += layer.weight_regularizer_l2 * \
        np.sum(layer.weights * \
        layer.weights)
      if layer.bias_regularizer_l1 > 0:
        regularization_loss += layer.bias_regularizer_l1 * \
        np.sum(np.abs(layer.biases))
      if layer.bias_regularizer_l2 > 0:
        regularization_loss += layer.bias_regularizer_l2 * \
        np.sum(layer.biases * \
        layer.biases)
    return regularization_loss
  def remember_trainable_layers(self, trainable_layers):
    self.trainable_layers = trainable_layers
  def calculate(self, output, y, *, include_regularization=False):
    sample_losses = self.forward(output, y)
    data_loss = np.mean(sample_losses)
    self.accumulated_sum += np.sum(sample_losses)
    self.accumulated_count += len(sample_losses)
    if not include_regularization:
      return data_loss
    return data_loss, self.regularization_loss()
  def calculate_accumulated(self, *, include_regularization=False):
    data_loss = self.accumulated_sum / self.accumulated_count
    if not include_regularization:
      return data_loss
    return data_loss, self.regularization_loss()
  def new_pass(self):
    self.accumulated_sum = 0
    self.accumulated_count = 0

In [3]:
class Optimizer_Adam:
  def __init__(self, learning_rate=0.001, decay=0., epsilon=1e-7,
  beta_1=0.9, beta_2=0.999):
    self.learning_rate = learning_rate
    self.current_learning_rate = learning_rate
    self.decay = decay
    self.iterations = 0
    self.epsilon = epsilon
    self.beta_1 = beta_1
    self.beta_2 = beta_2
  def pre_update_params(self):
    if self.decay:
      self.current_learning_rate = self.learning_rate * \
      (1. / (1. + self.decay * self.iterations))
  def update_params(self, layer):
    if not hasattr(layer, 'weight_cache'):
      layer.weight_momentums = np.zeros_like(layer.weights)
      layer.weight_cache = np.zeros_like(layer.weights)
      layer.bias_momentums = np.zeros_like(layer.biases)
      layer.bias_cache = np.zeros_like(layer.biases)
    layer.weight_momentums = self.beta_1 * \
    layer.weight_momentums + \
    (1 - self.beta_1) * layer.dweights
    layer.bias_momentums = self.beta_1 * \
    layer.bias_momentums + \
    (1 - self.beta_1) * layer.dbiases
    weight_momentums_corrected = layer.weight_momentums / \
    (1 - self.beta_1 ** (self.iterations + 1))
    bias_momentums_corrected = layer.bias_momentums / \
    (1 - self.beta_1 ** (self.iterations + 1))
    layer.weight_cache = self.beta_2 * layer.weight_cache + \
    (1 - self.beta_2) * layer.dweights**2
    layer.bias_cache = self.beta_2 * layer.bias_cache + \
    (1 - self.beta_2) * layer.dbiases**2
    weight_cache_corrected = layer.weight_cache / \
    (1 - self.beta_2 ** (self.iterations + 1))
    bias_cache_corrected = layer.bias_cache / \
    (1 - self.beta_2 ** (self.iterations + 1))
    layer.weights += -self.current_learning_rate * \
    weight_momentums_corrected / \
    (np.sqrt(weight_cache_corrected) +\
    self.epsilon)
    layer.biases += -self.current_learning_rate * \
    bias_momentums_corrected / \
    (np.sqrt(bias_cache_corrected) +\
    self.epsilon)
  def post_update_params(self):
    self.iterations += 1

class Loss_CategoricalCrossentropy(Loss):
  def forward(self, y_pred, y_true):
    samples = len(y_pred)
    y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)
    if len(y_true.shape) == 1:
      correct_confidences = y_pred_clipped[
      range(samples),
      y_true
      ]
    elif len(y_true.shape) == 2:
      correct_confidences = np.sum(
      y_pred_clipped * y_true,
      axis=1
      )
    negative_log_likelihoods = -np.log(correct_confidences)
    return negative_log_likelihoods
  def backward(self, dvalues, y_true):
    samples = len(dvalues)
    labels = len(dvalues[0])
    if len(y_true.shape) == 1:
      y_true = np.eye(labels)[y_true]
    self.dinputs = -y_true / dvalues
    self.dinputs = self.dinputs / samples

class Activation_Softmax_Loss_CategoricalCrossentropy():
  def backward(self, dvalues, y_true):
    samples = len(dvalues)
    if len(y_true.shape) == 2:
      y_true = np.argmax(y_true, axis=1)
    self.dinputs = dvalues.copy()
    self.dinputs[range(samples), y_true] -= 1
    self.dinputs = self.dinputs / samples

In [4]:
class Accuracy:
  def calculate(self, predictions, y):
    comparisons = self.compare(predictions, y)
    accuracy = np.mean(comparisons)
    self.accumulated_sum += np.sum(comparisons)
    self.accumulated_count += len(comparisons)
    return accuracy
  def calculate_accumulated(self):
    accuracy = self.accumulated_sum / self.accumulated_count
    return accuracy
  def new_pass(self):
    self.accumulated_sum = 0
    self.accumulated_count = 0

class Accuracy_Categorical(Accuracy):
  def init(self, y):
    pass
  def compare(self, predictions, y):
    if len(y.shape) == 2:
      y = np.argmax(y, axis=1)
    return predictions == y

In [5]:
class Activation_ReLU:
  def forward(self, inputs, training):
    self.inputs = inputs
    self.output = np.maximum(0, inputs)
  def backward(self, dvalues):
    self.dinputs = dvalues.copy()
    self.dinputs[self.inputs <= 0] = 0
  def predictions(self, outputs):
    return outputs

class Activation_Softmax:
  def forward(self, inputs, training):
    self.inputs = inputs
    exp_values = np.exp(inputs - np.max(inputs, axis=1,
    keepdims=True))
    probabilities = exp_values / np.sum(exp_values, axis=1,
    keepdims=True)
    self.output = probabilities
  def backward(self, dvalues):
    self.dinputs = np.empty_like(dvalues)
    for index, (single_output, single_dvalues) in \
    enumerate(zip(self.output, dvalues)):
      single_output = single_output.reshape(-1, 1)
      jacobian_matrix = np.diagflat(single_output) - \
      np.dot(single_output, single_output.T)
      self.dinputs[index] = np.dot(jacobian_matrix,
      single_dvalues)
  def predictions(self, outputs):
    return np.argmax(outputs, axis=1)

In [6]:
import torchvision
import torchvision.transforms as transforms

class Model:
    def __init__(self):
        self.layers = []
        self.softmax_classifier_output = None

    def add(self, layer):
        self.layers.append(layer)

    def set(self, *, loss, optimizer, accuracy):
        self.loss = loss
        self.optimizer = optimizer
        self.accuracy = accuracy

    def finalize(self):
        self.input_layer = Layer_Input()
        layer_count = len(self.layers)
        self.trainable_layers = []
        for i in range(layer_count):
            if i == 0:
                self.layers[i].prev = self.input_layer
                self.layers[i].next = self.layers[i+1]
            elif i < layer_count - 1:
                self.layers[i].prev = self.layers[i-1]
                self.layers[i].next = self.layers[i+1]
            else:
                self.layers[i].prev = self.layers[i-1]
                self.layers[i].next = self.loss
                self.output_layer_activation = self.layers[i]
            if hasattr(self.layers[i], 'weights'):
                self.trainable_layers.append(self.layers[i])

        self.loss.remember_trainable_layers(self.trainable_layers)
        if isinstance(self.layers[-1], Activation_Softmax) and isinstance(self.loss, Loss_CategoricalCrossentropy):
            self.softmax_classifier_output = Activation_Softmax_Loss_CategoricalCrossentropy()

    def train(self, X, y, *, epochs=1, batch_size=None, print_every=1, validation_data=None):
        self.accuracy.init(y)
        train_steps = 1
        if validation_data is not None:
            validation_steps = 1
            X_val, y_val = validation_data
        if batch_size is not None:
            train_steps = len(X) // batch_size
            if train_steps * batch_size < len(X):
                train_steps += 1
            if validation_data is not None:
                validation_steps = len(X_val) // batch_size
                if validation_steps * batch_size < len(X_val):
                    validation_steps += 1

        for epoch in range(1, epochs+1):
            print(f'epoch: {epoch}')
            self.loss.new_pass()
            self.accuracy.new_pass()
            for step in range(train_steps):
                if batch_size is None:
                    batch_X = X
                    batch_y = y
                else:
                    batch_X = X[step*batch_size:(step+1)*batch_size]
                    batch_y = y[step*batch_size:(step+1)*batch_size]
                output = self.forward(batch_X, training=True)
                data_loss, regularization_loss = self.loss.calculate(output, batch_y, include_regularization=True)
                loss = data_loss + regularization_loss
                predictions = self.output_layer_activation.predictions(output)
                accuracy = self.accuracy.calculate(predictions, batch_y)
                self.backward(output, batch_y)
                self.optimizer.pre_update_params()
                for layer in self.trainable_layers:
                    self.optimizer.update_params(layer)
                self.optimizer.post_update_params()
                if not step % print_every or step == train_steps - 1:
                    print(f'step: {step}, acc: {accuracy:.3f}, loss: {loss:.3f} (data_loss: {data_loss:.3f}, reg_loss: {regularization_loss:.3f}), lr: {self.optimizer.current_learning_rate}')

            epoch_data_loss, epoch_regularization_loss = self.loss.calculate_accumulated(include_regularization=True)
            epoch_loss = epoch_data_loss + epoch_regularization_loss
            epoch_accuracy = self.accuracy.calculate_accumulated()
            print(f'training, acc: {epoch_accuracy:.3f}, loss: {epoch_loss:.3f} (data_loss: {epoch_data_loss:.3f}, reg_loss: {epoch_regularization_loss:.3f}), lr: {self.optimizer.current_learning_rate}')

            if validation_data is not None:
                self.loss.new_pass()
                self.accuracy.new_pass()
                for step in range(validation_steps):
                    if batch_size is None:
                        batch_X = X_val
                        batch_y = y_val
                    else:
                        batch_X = X_val[step*batch_size:(step+1)*batch_size]
                        batch_y = y_val[step*batch_size:(step+1)*batch_size]
                    output = self.forward(batch_X, training=False)
                    self.loss.calculate(output, batch_y)
                    predictions = self.output_layer_activation.predictions(output)
                    self.accuracy.calculate(predictions, batch_y)
                validation_loss = self.loss.calculate_accumulated()
                validation_accuracy = self.accuracy.calculate_accumulated()
                print(f'validation, acc: {validation_accuracy:.3f}, loss: {validation_loss:.3f}')

    def forward(self, X, training):
        self.input_layer.forward(X, training)
        for layer in self.layers:
            layer.forward(layer.prev.output, training)
        return layer.output

    def backward(self, output, y):
        if self.softmax_classifier_output is not None:
            self.softmax_classifier_output.backward(output, y)
            self.layers[-1].dinputs = self.softmax_classifier_output.dinputs
            for layer in reversed(self.layers[:-1]):
                layer.backward(layer.next.dinputs)
            return
        self.loss.backward(output, y)
        for layer in reversed(self.layers):
            layer.backward(layer.next.dinputs)

try:
    train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
    test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True)

    X = train_dataset.data.numpy()
    y = train_dataset.targets.numpy().astype(np.uint8)
    X_test = test_dataset.data.numpy()
    y_test = test_dataset.targets.numpy().astype(np.uint8)

    keys = np.array(range(X.shape[0]))
    np.random.shuffle(keys)
    X = X[keys]
    y = y[keys]

    X = (X.reshape(X.shape[0], -1).astype(np.float32) - 127.5) / 127.5
    X_test = (X_test.reshape(X_test.shape[0], -1).astype(np.float32) - 127.5) / 127.5

    model = Model()
    model.add(Layer_Dense(X.shape[1], 128))
    model.add(Activation_ReLU())
    model.add(Layer_Dense(128, 128))
    model.add(Activation_ReLU())
    model.add(Layer_Dense(128, 10))
    model.add(Activation_Softmax())

    model.set(
        loss=Loss_CategoricalCrossentropy(),
        optimizer=Optimizer_Adam(decay=1e-4),
        accuracy=Accuracy_Categorical()
    )

    model.finalize()
    model.train(X, y, validation_data=(X_test, y_test), epochs=10, batch_size=128, print_every=100)
except Exception as e:
    print(f"An error occurred during dataset loading or training: {e}")

100%|██████████| 26.4M/26.4M [00:02<00:00, 9.76MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 141kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.69MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.9MB/s]


epoch: 1
step: 0, acc: 0.070, loss: 2.302 (data_loss: 2.302, reg_loss: 0.000), lr: 0.001
step: 100, acc: 0.773, loss: 0.608 (data_loss: 0.608, reg_loss: 0.000), lr: 0.0009900990099009901
step: 200, acc: 0.844, loss: 0.430 (data_loss: 0.430, reg_loss: 0.000), lr: 0.000980392156862745
step: 300, acc: 0.820, loss: 0.592 (data_loss: 0.592, reg_loss: 0.000), lr: 0.0009708737864077671
step: 400, acc: 0.883, loss: 0.317 (data_loss: 0.317, reg_loss: 0.000), lr: 0.0009615384615384615
step: 468, acc: 0.844, loss: 0.472 (data_loss: 0.472, reg_loss: 0.000), lr: 0.0009552923194497518
training, acc: 0.760, loss: 0.649 (data_loss: 0.649, reg_loss: 0.000), lr: 0.0009552923194497518
validation, acc: 0.826, loss: 0.484
epoch: 2
step: 0, acc: 0.836, loss: 0.479 (data_loss: 0.479, reg_loss: 0.000), lr: 0.0009552010698251983
step: 100, acc: 0.852, loss: 0.363 (data_loss: 0.363, reg_loss: 0.000), lr: 0.0009461633077869241
step: 200, acc: 0.922, loss: 0.234 (data_loss: 0.234, reg_loss: 0.000), lr: 0.00093729

# New section